# FEATURE ENGINEERING

In this notebook I will cover the feature engineering aspect of the project. This will include:
- Balance drain ratio
- Cyclical time encoding
- Derived behavioural features documented with rationale

In [53]:
# Library Importation
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [54]:
# data loading
df = pd.read_csv('../Data/mpesa_synthetic.csv')
df.head()

,transaction_id,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,transaction_type,hour,month_2026,day_of_week,device_type,region,is_fraud
0,UA0IFD0TV,703.90,58586.32,57882.42,29932.92,30636.82,peer,14,1,Tuesday,smartphone,Nairobi,0
1,UJAHXTHV3,254.44,8088.00,7833.56,22962.44,23216.88,peer,18,10,Saturday,feature,Eldoret,1
2,UEF8MDD4V,609.04,56675.00,56065.96,1029.22,1638.26,till,7,5,Thursday,smartphone,Kisumu,0
3,UBT3W5UZB,5255.34,75090.36,69835.02,38.94,5294.28,paybill,11,2,Monday,smartphone,Nakuru,0
4,UGKWNNHJ7,7282.67,24408.96,17126.29,26237.82,33520.49,till,0,7,Saturday,smartphone,Nairobi,0


In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           120000 non-null  str    
 1   amount                   120000 non-null  float64
 2   sender_balance_before    120000 non-null  float64
 3   sender_balance_after     120000 non-null  float64
 4   receiver_balance_before  120000 non-null  float64
 5   receiver_balance_after   120000 non-null  float64
 6   transaction_type         120000 non-null  str    
 7   hour                     120000 non-null  int64  
 8   month_2026               120000 non-null  int64  
 9   day_of_week              120000 non-null  str    
 10  device_type              120000 non-null  str    
 11  region                   120000 non-null  str    
 12  is_fraud                 120000 non-null  int64  
dtypes: float64(5), int64(3), str(5)
memory usage: 11.9 MB


In [56]:
zero_transactions = df[df['amount'] == 0]
zero_transactions

,transaction_id,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,transaction_type,hour,month_2026,day_of_week,device_type,region,is_fraud
84990,UDSI0IDKI,0.0,5827.75,5827.75,23707.86,23707.86,peer,19,4,Sunday,feature,Eldoret,0


In [57]:
# Droping the zero transaction entry
df = df.drop(zero_transactions.index)
df.shape

(119999, 13)

In [58]:
#Mapping day of week to number
# craeting a function to map all entries in the day of week column
mapper = {
    'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 'Thursday': 4,
    'Friday': 5, 'Saturday': 6, 'Sunday': 7
}

df['day'] = df['day_of_week'].map(mapper)
df['day'].value_counts()

day
1    17408
5    17339
7    17189
6    17142
2    17110
3    16989
4    16822
Name: count, dtype: int64

In [59]:
# Mapping is_farud to transaction_status(Labled class category)
def status_mapp(status):
    if status == 0:
        return 'LEGITIMATE'
    else:
        return 'FRAUD'

df['transaction_status'] = df['is_fraud'].apply(lambda x: status_mapp(x))
df['transaction_status'].value_counts()

transaction_status
LEGITIMATE    116489
FRAUD           3510
Name: count, dtype: int64

In [61]:
under_1 = df[df['amount'] < 1]
under_1.shape

(84, 15)

In [63]:
fraud_under1 = under_1[under_1['transaction_status'] == 'FRAUD']
fraud_under1.shape

(2, 15)

In [64]:
# Droping transactions below KES 1
df = df.drop(under_1.index)
df.shape

(119915, 15)

> The reason as to why I droped transactions below **KES 1** is because in a real world setting the minimum transactable amount is **KES 1**. The final deployed model should be able to perform in real world settings and training it with impossible noise only overfits the model for no reason.